# 第12章: PyTorch によるニューラルネットワーク学習

この Notebook は、原本 `machine-learning-book/ch12/ch12_part1.ipynb` と
`machine-learning-book/ch12/ch12_part2.ipynb` を、最新の Python 環境と
`pytest --nbmake` による CI 実行向けに 1 本へ再構成したものです。

原本の教育意図を保ちながら、次の点を更新しています。

- Notebook 拡張や外部ダウンロードには依存せず、ローカルにある図版・画像・scikit-learn データセットで完結させる
- `torchvision` 依存を必須にせず、PyTorch のテンソル、DataLoader、モデル定義、保存再読込を確認できる構成にする
- 線形回帰と Iris 分類を、CI で現実的な時間に収まる小規模な学習ループで検証する


## この Notebook で確認すること

- 原本図版とローカル画像を `src/` 配下から安定して参照できることを確認する
- PyTorch テンソルの作成、形状操作、数値演算、連結操作を確認する
- `TensorDataset` / `DataLoader` とカスタム `Dataset` を使った入力パイプラインを構築する
- PyTorch の autograd と `torch.nn` / `torch.optim` による線形回帰を確認する
- Iris データセットに対する MLP 分類、評価、`state_dict` の保存再読込を確認する
- シグモイド、ソフトマックス、tanh、ReLU の振る舞いを比較する


In [ ]:
from importlib.metadata import version
from pathlib import Path
import platform
import sys
import tempfile

from IPython.display import Image as IPythonImage, display
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, TensorDataset


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "machine-learning-book").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("リポジトリルートを見つけられませんでした。")


REPO_ROOT = find_repo_root()
CHAPTER_DIR = REPO_ROOT / "machine-learning-book" / "ch12"
FIGURE_DIR = CHAPTER_DIR / "figures"
IMAGE_DIR = CHAPTER_DIR / "cat_dog_images"

assert FIGURE_DIR.exists(), f"図版ディレクトリが見つかりません: {FIGURE_DIR}"
assert IMAGE_DIR.exists(), f"画像ディレクトリが見つかりません: {IMAGE_DIR}"

torch.set_num_threads(1)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python 実行ファイル: {sys.executable}")
print(f"Python バージョン: {platform.python_version()}")
print(f"Matplotlib バックエンド: {matplotlib.get_backend()}")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {DEVICE}")
print(f"Chapter directory: {CHAPTER_DIR}")


In [ ]:
package_versions = pd.DataFrame(
    [
        ("numpy", version("numpy")),
        ("pandas", version("pandas")),
        ("matplotlib", version("matplotlib")),
        ("scikit-learn", version("scikit-learn")),
        ("torch", version("torch")),
        ("pytest", version("pytest")),
    ],
    columns=["パッケージ", "バージョン"],
)
package_versions


## 原本図版の参照

章の導入で使われている図版を読み込み、移行版 Notebook から原本アセットにアクセスできることを確認します。


In [ ]:
for figure_name in ["12_01.png", "12_02.png", "12_11.png"]:
    print(figure_name)
    display(IPythonImage(filename=str(FIGURE_DIR / figure_name), width=560))


## PyTorch テンソルの基本

テンソル作成、dtype と shape の変更、数学演算、分割・結合といった原本前半の基本操作を確認します。


In [ ]:
torch.manual_seed(1)
np.set_printoptions(precision=3, suppress=True)

a = [1, 2, 3]
b = np.array([4, 5, 6], dtype=np.int32)
t_a = torch.tensor(a)
t_b = torch.from_numpy(b)
t_ones = torch.ones(2, 3)
t_rand = torch.rand(2, 3)
t_transposed = torch.transpose(torch.rand(3, 5), 0, 1)
t_chunk = torch.chunk(torch.arange(6, dtype=torch.float32), 3)
t_stack = torch.stack([torch.ones(3), torch.zeros(3)], dim=1)

display(
    pd.Series(
        {
            "is_tensor(a)": torch.is_tensor(a),
            "is_tensor(t_a)": torch.is_tensor(t_a),
            "t_a_dtype": str(t_a.dtype),
            "t_b_dtype": str(t_b.dtype),
            "t_ones_shape": str(tuple(t_ones.shape)),
            "t_transposed_shape": str(tuple(t_transposed.shape)),
        }
    ).to_frame(name="値")
)

display(pd.DataFrame(t_rand.numpy(), columns=["c0", "c1", "c2"]).round(3))
display(pd.DataFrame({"chunk_0": t_chunk[0].numpy(), "chunk_1": t_chunk[1].numpy(), "chunk_2": t_chunk[2].numpy()}))
pd.DataFrame(t_stack.numpy(), columns=["stack_col0", "stack_col1"])


## DataLoader とカスタム Dataset

`TensorDataset` と `DataLoader` による基本的なバッチ化と、ローカル画像を扱うカスタム `Dataset` を確認します。


In [ ]:
torch.manual_seed(1)

t_x = torch.rand((4, 3), dtype=torch.float32)
t_y = torch.arange(4)
joint_dataset = TensorDataset(t_x, t_y)
data_loader = DataLoader(joint_dataset, batch_size=2, shuffle=True)
example_batches = list(data_loader)

display(
    pd.DataFrame(
        {
            "batch_index": [1, 2],
            "x_shape": [str(tuple(batch[0].shape)) for batch in example_batches],
            "y_values": [batch[1].tolist() for batch in example_batches],
        }
    )
)


class CatDogImageDataset(Dataset):
    def __init__(self, image_dir: Path):
        self.file_list = sorted(image_dir.glob("*.jpg"))

    def __len__(self) -> int:
        return len(self.file_list)

    def __getitem__(self, idx: int):
        path = self.file_list[idx]
        image = Image.open(path).convert("RGB").resize((80, 80))
        image_np = np.asarray(image, dtype=np.float32) / 255.0
        image_tensor = torch.from_numpy(image_np).permute(2, 0, 1)
        label = 0 if path.name.startswith("cat") else 1
        return image_tensor, label, path.name


image_dataset = CatDogImageDataset(IMAGE_DIR)
sample_image, sample_label, sample_name = image_dataset[0]

fig, axes = plt.subplots(2, 3, figsize=(7.2, 4.8), sharex=True, sharey=True)
for ax, (image_tensor, label, name) in zip(axes.flatten(), [image_dataset[i] for i in range(len(image_dataset))]):
    ax.imshow(image_tensor.permute(1, 2, 0).numpy())
    ax.set_title(f"{name}\nlabel={label}")
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()
plt.close(fig)

pd.Series(
    {
        "num_images": len(image_dataset),
        "sample_name": sample_name,
        "sample_label": sample_label,
        "sample_tensor_shape": str(tuple(sample_image.shape)),
    }
).to_frame(name="値")


## PyTorch での線形回帰

原本の流れに沿って、まずは autograd を直接使った線形回帰、その後 `nn.Linear` と `optim.SGD` を使った実装を確認します。


In [ ]:
X_train_lin = np.arange(10, dtype="float32").reshape((10, 1))
y_train_lin = np.array([1.0, 1.3, 3.1, 2.0, 5.0, 6.3, 6.6, 7.4, 8.0, 9.0], dtype="float32")

X_train_lin_norm = (X_train_lin - np.mean(X_train_lin)) / np.std(X_train_lin)
X_train_lin_norm = torch.from_numpy(X_train_lin_norm)
y_train_lin_t = torch.from_numpy(y_train_lin)

train_ds_lin = TensorDataset(X_train_lin_norm, y_train_lin_t)
train_dl_lin = DataLoader(train_ds_lin, batch_size=1, shuffle=True)

torch.manual_seed(1)
weight = torch.randn(1, requires_grad=True)
bias = torch.zeros(1, requires_grad=True)


def loss_fn(input: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    return (input - target).pow(2).mean()


def model_fn(xb: torch.Tensor) -> torch.Tensor:
    return xb @ weight + bias


for epoch in range(120):
    for x_batch, y_batch in train_dl_lin:
        pred = model_fn(x_batch)
        loss = loss_fn(pred, y_batch)
        loss.backward()
        with torch.no_grad():
            weight -= weight.grad * 0.01
            bias -= bias.grad * 0.01
            weight.grad.zero_()
            bias.grad.zero_()

manual_params = {"weight": float(weight.item()), "bias": float(bias.item())}

torch.manual_seed(1)
nn_model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(nn_model.parameters(), lr=0.01)
mse_loss = nn.MSELoss()

for epoch in range(120):
    for x_batch, y_batch in train_dl_lin:
        pred = nn_model(x_batch)[:, 0]
        loss = mse_loss(pred, y_batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

nn_params = {"weight": float(nn_model.weight.item()), "bias": float(nn_model.bias.item())}

X_test_lin = np.linspace(0, 9, num=100, dtype="float32").reshape(-1, 1)
X_test_lin_norm = (X_test_lin - np.mean(X_train_lin)) / np.std(X_train_lin)
X_test_lin_norm_t = torch.from_numpy(X_test_lin_norm)
y_pred_lin = nn_model(X_test_lin_norm_t).detach().numpy()

fig, ax = plt.subplots(figsize=(5.4, 3.5))
ax.plot(X_train_lin_norm.numpy(), y_train_lin_t.numpy(), "o", markersize=8, label="Training examples")
ax.plot(X_test_lin_norm, y_pred_lin, "--", linewidth=2, label="Linear reg.")
ax.set_xlabel("x (normalized)")
ax.set_ylabel("y")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()
plt.close(fig)

pd.DataFrame([manual_params, nn_params], index=["manual_autograd", "nn_linear"]).round(4)


## Iris 分類用の MLP

Iris データセットを使って、`nn.Module` ベースの多層パーセプトロンを学習します。
出力層にはソフトマックスを入れず、`CrossEntropyLoss` にロジットを渡す現行 PyTorch の一般的な構成にしています。


In [ ]:
iris = load_iris()
X = iris["data"].astype(np.float32)
y = iris["target"].astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=1.0 / 3.0,
    random_state=1,
    stratify=y,
)

train_mean = X_train.mean(axis=0)
train_std = X_train.std(axis=0)
X_train_norm = (X_train - train_mean) / train_std
X_test_norm = (X_test - train_mean) / train_std

X_train_t = torch.from_numpy(X_train_norm).float()
y_train_t = torch.from_numpy(y_train)
X_test_t = torch.from_numpy(X_test_norm).float()
y_test_t = torch.from_numpy(y_test)

train_ds = TensorDataset(X_train_t, y_train_t)
train_dl = DataLoader(train_ds, batch_size=8, shuffle=True)


class IrisMLP(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, output_size: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


torch.manual_seed(1)
iris_model = IrisMLP(input_size=X_train_t.shape[1], hidden_size=16, output_size=3)
optimizer = torch.optim.Adam(iris_model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

num_epochs = 80
loss_hist = []
accuracy_hist = []

for epoch in range(num_epochs):
    epoch_loss = 0.0
    epoch_correct = 0
    for x_batch, y_batch in train_dl:
        logits = iris_model(x_batch)
        loss = loss_fn(logits, y_batch.long())
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        epoch_loss += loss.item() * y_batch.size(0)
        epoch_correct += (torch.argmax(logits, dim=1) == y_batch).sum().item()

    loss_hist.append(epoch_loss / len(train_dl.dataset))
    accuracy_hist.append(epoch_correct / len(train_dl.dataset))

with torch.inference_mode():
    test_logits = iris_model(X_test_t)
    test_pred = torch.argmax(test_logits, dim=1)
    test_acc = (test_pred == y_test_t).float().mean().item()

fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.2))
axes[0].plot(loss_hist, linewidth=2)
axes[0].set_title("Training loss")
axes[0].set_xlabel("Epoch")
axes[0].grid(alpha=0.3)
axes[1].plot(accuracy_hist, linewidth=2)
axes[1].set_title("Training accuracy")
axes[1].set_xlabel("Epoch")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()
plt.close(fig)

pd.Series(
    {
        "final_train_loss": round(loss_hist[-1], 4),
        "final_train_accuracy": round(accuracy_hist[-1], 4),
        "test_accuracy": round(test_acc, 4),
    }
).to_frame(name="値")


## モデル保存と再読込

`state_dict` を一時ファイルへ保存し、新しいモデルインスタンスへ読み戻して同じ予測精度が得られることを確認します。


In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    save_path = Path(tmpdir) / "iris_classifier_state.pt"
    torch.save(iris_model.state_dict(), save_path)

    reloaded_model = IrisMLP(input_size=X_train_t.shape[1], hidden_size=16, output_size=3)
    reloaded_model.load_state_dict(torch.load(save_path, map_location="cpu"))
    reloaded_model.eval()

    with torch.inference_mode():
        reloaded_logits = reloaded_model(X_test_t)
        reloaded_pred = torch.argmax(reloaded_logits, dim=1)
        reloaded_acc = (reloaded_pred == y_test_t).float().mean().item()

pd.Series(
    {
        "saved_file_exists_during_run": True,
        "reloaded_test_accuracy": round(reloaded_acc, 4),
        "predictions_match": bool(torch.equal(test_pred, reloaded_pred)),
    }
).to_frame(name="値")


## 活性化関数の比較

ロジスティック関数、ソフトマックス、tanh、ReLU の出力を同じ入力に対して比較し、章後半の要点を確認します。


In [ ]:
x_vals = torch.linspace(-3, 3, steps=7)
logistic_vals = torch.sigmoid(x_vals)
tanh_vals = torch.tanh(x_vals)
relu_vals = torch.relu(x_vals)
softmax_input = torch.tensor([[1.1, 1.2, 0.8, 0.4], [0.2, 0.4, 1.0, 0.2], [0.6, 1.5, 1.2, 0.7]])
softmax_vals = torch.softmax(softmax_input[:, 0], dim=0)

activation_df = pd.DataFrame(
    {
        "x": x_vals.numpy(),
        "sigmoid": logistic_vals.numpy(),
        "tanh": tanh_vals.numpy(),
        "relu": relu_vals.numpy(),
    }
)

display(activation_df.round(4))
pd.Series(
    {
        "softmax_input_first_column": softmax_input[:, 0].tolist(),
        "softmax_output": [round(float(v), 4) for v in softmax_vals],
        "softmax_sum": round(float(softmax_vals.sum()), 4),
    }
).to_frame(name="値")


## まとめ

この移行版 Notebook では、第12章の中心となる PyTorch テンソル操作、DataLoader、カスタム Dataset、
線形回帰、Iris 用 MLP、モデル保存再読込、活性化関数の比較を、最新の PyTorch API で再構成しました。

原本の `torchvision` や外部ダウンロード依存を外しているため、CI 上でも継続的に PyTorch 章の主要コードパスを検証できます。
